# D-01 EDA - Migrants by Place of Birth (Census 2011)

**File:** `DS-0000-D01-MDDS.XLSX`  
**What this has:** For each destination state / UT, this table shows where migrants were born, along with total, male, female, rural, and urban counts.

This notebook rebuilds the state-wise `D01_cleaned.csv` used by the dashboard.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

INPUT_FILE = 'DS-0000-D01-MDDS.XLSX'
SKIP_ROWS = 4

COL_NAMES = [
    'TableName', 'StateCode', 'DistrictCode', 'AreaName', 'BirthPlace',
    'Total_Persons', 'Total_Males', 'Total_Females',
    'Rural_Persons', 'Rural_Males', 'Rural_Females',
    'Urban_Persons', 'Urban_Males', 'Urban_Females',
]

df_raw = pd.read_excel(INPUT_FILE, skiprows=SKIP_ROWS, header=None)
df_raw.columns = COL_NAMES[:df_raw.shape[1]]
print(f'Raw shape: {df_raw.shape}')
df_raw.head(8)


## 1. Raw snapshot


In [ ]:
print(df_raw.dtypes)
print()
print(df_raw.isnull().sum())


In [ ]:
num_cols = [c for c in df_raw.columns if c not in ('TableName','StateCode','DistrictCode','AreaName','BirthPlace')]
df_raw[num_cols].describe()


## 2. Understanding subtotal rows


In [ ]:
bp_series = df_raw['BirthPlace'].dropna().astype(str).str.strip()

SUMMARY_BP = [
    'Total Population','Born within India','Within the state of enumeration',
    'Born in the place of enumeration','Born elsewhere in the district of enumeration',
    'Born in other districts of the state',
    'States in India beyond the state of enumeration',
    'Born Outside India','Countries in Asia beyond India','Countries in Europe',
    'Countries in Africa','Countries in the Americas','Countries in Oceania',
    'Elsewhere','Unclassifiable'
]

print('BirthPlace values:')
for v in sorted(bp_series.unique()):
    n = (bp_series == v).sum()
    note = "  <- subtotal" if v in SUMMARY_BP else ("  <- header row" if v.isdigit() else "")
    print(f'  {n:>4}  {v}{note}')


In [ ]:
an_series = df_raw['AreaName'].dropna().astype(str).str.strip()
print('AreaName values:')
for v in sorted(an_series.unique()):
    n = (an_series == v).sum()
    note = "  <- national total, remove" if v == "INDIA" else ""
    print(f'  {n:>4}  {v}{note}')


## 3. Tier checks on India totals


In [ ]:
india = df_raw[df_raw['AreaName'].astype(str).str.strip() == 'INDIA'].copy()
for c in ['Total_Persons', 'Rural_Persons', 'Urban_Persons']:
    india[c] = pd.to_numeric(india[c], errors='coerce')

def iv(bp_name):
    row = india[india['BirthPlace'].astype(str).str.strip() == bp_name]
    return float(row['Total_Persons'].values[0]) if len(row) else None

total = iv('Total Population')
b_india = iv('Born within India')
b_out = iv('Born Outside India')
w_state = iv('Within the state of enumeration')
beyond = iv('States in India beyond the state of enumeration')

print(f'Total Population : {total:>15,.0f}')
print(f'Born in India    : {b_india:>15,.0f}')
print(f'Born Outside     : {b_out:>15,.0f}')
print(f'Sum              : {b_india+b_out:>15,.0f}  ok = {abs(total - b_india - b_out) < 1}')
print()
print(f'Within state     : {w_state:>15,.0f}')
print(f'Beyond state     : {beyond:>15,.0f}')
print(f'Sum              : {w_state+beyond:>15,.0f}  ok = {abs(b_india - w_state - beyond) < 1}')


## 4. Cleaning


In [ ]:
df = df_raw.copy()

df = df[~df['BirthPlace'].astype(str).str.strip().str.match(r'^\d+$')]
df = df.dropna(subset=['AreaName', 'BirthPlace'])
df = df[df['AreaName'].astype(str).str.strip() != 'INDIA']

SUMMARY_BP = {
    'Total Population', 'Born within India',
    'Within the state of enumeration',
    'Born in the place of enumeration',
    'Born elsewhere in the district of enumeration',
    'Born in other districts of the state',
    'States in India beyond the state of enumeration',
    'Born Outside India', 'Countries in Asia beyond India',
    'Countries in Europe', 'Countries in Africa',
    'Countries in the Americas', 'Countries in Oceania',
    'Elsewhere', 'Unclassifiable',
}
df = df[~df['BirthPlace'].astype(str).str.strip().isin(SUMMARY_BP)]

num_cols = [c for c in df.columns if c not in ('TableName','StateCode','DistrictCode','AreaName','BirthPlace')]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

df = df[df['Total_Persons'] > 0]

df['AreaName'] = df['AreaName'].astype(str).str.replace(r'^State\s*-\s*', '', regex=True).str.replace(r'\s*\(\d+\)\s*$', '', regex=True).str.strip()
df['BirthPlace'] = df['BirthPlace'].astype(str).str.strip()

print(f'Rows: {len(df)}')
print(f"States: {df['AreaName'].nunique()},  BirthPlaces: {df['BirthPlace'].nunique()}")


## 5. Validation checks


In [ ]:
dupes = df.duplicated(subset=['AreaName', 'BirthPlace'])
print(f'Duplicate (state, birthplace) pairs: {dupes.sum()}')

mf_bad = (abs(df['Total_Persons'] - df['Total_Males'] - df['Total_Females']) > 1).sum()
print(f'Rows where Total != M+F: {mf_bad}')

ru_bad = (abs(df['Total_Persons'] - df['Rural_Persons'] - df['Urban_Persons']) > 1).sum()
print(f'Rows where Total != Rural+Urban: {ru_bad}')

sample = df['AreaName'].iloc[0]
raw_tot_row = df_raw[
    (df_raw['AreaName'].astype(str).str.contains(sample[:10], case=False, na=False)) &
    (df_raw['BirthPlace'].astype(str).str.strip() == 'Total Population')
]
if len(raw_tot_row):
    raw_tot = int(pd.to_numeric(raw_tot_row['Total_Persons'].values[0], errors='coerce'))
    clean_sum = df[df['AreaName'] == sample]['Total_Persons'].sum()
    print(f'\n{sample}: raw reported total = {raw_tot:,}, our cleaned sum = {clean_sum:,} ({clean_sum/raw_tot*100:.1f}%)')


## 6. Quick EDA


In [ ]:
print(f"States / UTs   : {df['AreaName'].nunique()}")
print(f"BirthPlaces    : {df['BirthPlace'].nunique()}")
print(f'Total rows     : {len(df):,}')
print(f"Total persons  : {df['Total_Persons'].sum():,}")

rps = df.groupby('AreaName').size()
print(f'Rows per state  min={rps.min()}, max={rps.max()}, mean={rps.mean():.1f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dest = df.groupby('AreaName')['Total_Persons'].sum().sort_values(ascending=False).head(10)
axes[0].barh(dest.index[::-1], dest.values[::-1] / 1e6, color='steelblue')
axes[0].set_xlabel('Migrants (millions)')
axes[0].set_title('Top 10 Destination States (D01)')
axes[0].xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f M'))

src = df.groupby('BirthPlace')['Total_Persons'].sum().sort_values(ascending=False).head(10)
axes[1].barh(src.index[::-1], src.values[::-1] / 1e6, color='coral')
axes[1].set_xlabel('Persons (millions)')
axes[1].set_title('Top 10 Birth Places')
axes[1].xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f M'))

plt.tight_layout()
plt.show()


In [ ]:
state_sr = df.groupby('AreaName').agg(Males=('Total_Males','sum'), Females=('Total_Females','sum'))
state_sr['SexRatio'] = (state_sr['Females'] / state_sr['Males'] * 1000).round(0)
state_sr = state_sr.sort_values('SexRatio')

colors = ['#e74c3c' if sr < 1000 else '#2ecc71' for sr in state_sr['SexRatio']]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(state_sr.index, state_sr['SexRatio'], color=colors)
ax.axvline(1000, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Females per 1000 Males')
ax.set_title('Sex Ratio by Destination State (D01)')
plt.tight_layout()
plt.show()

overall = state_sr['Females'].sum() / state_sr['Males'].sum() * 1000
print(f'Overall sex ratio: {overall:.0f} F per 1000 M')


In [ ]:
indian_states = set(df['AreaName'].unique())
df_corr = df[df['BirthPlace'].isin(indian_states)].copy()
df_corr['Corridor'] = df_corr['BirthPlace'] + ' -> ' + df_corr['AreaName']

top_corr = df_corr.nlargest(12, 'Total_Persons')[['Corridor','Total_Persons']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_corr['Corridor'][::-1].values, top_corr['Total_Persons'][::-1].values / 1e5, color='mediumpurple')
ax.set_xlabel('Persons (hundred thousands)')
ax.set_title('Top 12 Interstate Migration Corridors (D01)')
plt.tight_layout()
plt.show()


In [ ]:
bp_totals = df.groupby('BirthPlace')['Total_Persons'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(bp_totals)), bp_totals.values, color='darkorange', width=0.8)
ax.set_yscale('log')
ax.set_xlabel('BirthPlace (ranked high to low)')
ax.set_ylabel('Persons (log scale)')
ax.set_title('Distribution of BirthPlace totals - D01')
ax.set_xticks([])
plt.tight_layout()
plt.show()

print(f'Top    : {bp_totals.index[0]}  ({bp_totals.values[0]:,})')
print(f'Bottom : {bp_totals.index[-1]}  ({bp_totals.values[-1]:,})')


## 7. Export cleaned file


In [ ]:
keep_cols = ['AreaName','BirthPlace','Total_Persons','Total_Males','Total_Females',
             'Rural_Persons','Rural_Males','Rural_Females',
             'Urban_Persons','Urban_Males','Urban_Females']
df_final = df[[c for c in keep_cols if c in df.columns]].reset_index(drop=True)

print(f'Shape  : {df_final.shape}')
print(f'Nulls  : {df_final.isnull().sum().sum()}')
print(f"Total  : {df_final['Total_Persons'].sum():,}")

df_final.to_csv('D01_cleaned.csv', index=False)
print('Saved -> D01_cleaned.csv')
df_final.head(10)
